# RAG System Evaluation Analysis

## 4-Way Comparison
1. **Baseline**: Current SmartEnhancedRAG with old dataset
2. **Baseline + Updated Data**: SmartEnhancedRAG with cleaned/fresh dataset
3. **MADAM-RAG + Old Data**: Multi-agent debate with old dataset
4. **MADAM-RAG + New Data**: Multi-agent debate with cleaned/fresh dataset

## Metrics
- **Accuracy**: % of correct answers
- **Precision**: Relevance of retrieved chunks
- **Recall**: Coverage of relevant information
- **F1 Score**: Harmonic mean of Precision & Recall
- **Confident Wrong**: False positives with high confidence
- **Response Time**: Average time to generate answer
- **Token Usage**: Computational cost

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import os

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

# Check current directory
print(f"Current directory: {os.getcwd()}")
print(f"Files in evaluation/: {os.listdir('.')[:10]}")

# Load scored results
csv_file = 'demo_scored_results.csv'
if not os.path.exists(csv_file):
    csv_file = 'evaluation/demo_scored_results.csv'

df = pd.read_csv(csv_file)

print(f"\n📊 Loaded {len(df)} scored results")
print(f"Columns: {list(df.columns)}")
df.head()

## 1. Load Logs from All Experiments

In [ ]:
def load_experiment_logs(log_file: str) -> pd.DataFrame:
    """
    Load JSONL logs and convert to DataFrame.
    """
    logs = []
    with open(log_file, 'r', encoding='utf-8') as f:
        for line in f:
            logs.append(json.loads(line))
    
    # Flatten nested structure
    flattened = []
    for log in logs:
        flat = {
            'query_id': log['query_id'],
            'query_text': log['query_text'],
            'experiment': log['experiment_name'],
            'timestamp': log['timestamp'],
            
            # Retrieval metrics
            'precision': log.get('retrieval', {}).get('precision'),
            'recall': log.get('retrieval', {}).get('recall'),
            'f1_score': log.get('retrieval', {}).get('f1_score'),
            'num_retrieved': log.get('retrieval', {}).get('num_retrieved'),
            
            # Response metrics
            'response_text': log['response']['text'],
            'model_name': log['response']['model_name'],
            'is_correct': log['response'].get('is_correct'),
            'confidence_score': log['response'].get('confidence_score'),
            'confident_wrong': log['response'].get('confident_wrong'),
            'response_time': log['response'].get('response_time_seconds'),
            'prompt_tokens': log['response'].get('token_usage', {}).get('prompt_tokens'),
            'completion_tokens': log['response'].get('token_usage', {}).get('completion_tokens'),
            'total_tokens': log['response'].get('token_usage', {}).get('total_tokens'),
            
            # MADAM-RAG specific
            'num_debate_rounds': log.get('madam_debate', {}).get('num_rounds'),
            'converged': log.get('madam_debate', {}).get('converged'),
        }
        flattened.append(flat)
    
    return pd.DataFrame(flattened)

# Load all experiments
experiments = {
    'baseline': 'evaluation/logs/baseline.jsonl',
    'baseline_updated': 'evaluation/logs/baseline_updated_data.jsonl',
    'madam_old': 'evaluation/logs/madam_rag_old_data.jsonl',
    'madam_new': 'evaluation/logs/madam_rag_new_data.jsonl',
}

dfs = {}
for name, log_file in experiments.items():
    if Path(log_file).exists():
        dfs[name] = load_experiment_logs(log_file)
        print(f"✅ Loaded {len(dfs[name])} queries from {name}")
    else:
        print(f"⚠️  Log file not found: {log_file}")

# Combine all dataframes
if dfs:
    df_all = pd.concat(dfs.values(), ignore_index=True)
    print(f"\n📊 Total queries across all experiments: {len(df_all)}")
else:
    print("❌ No log files found. Run evaluations first.")

## 2. Calculate Aggregate Metrics

In [ ]:
def calculate_metrics(df: pd.DataFrame) -> Dict:
    """
    Calculate aggregate metrics for an experiment.
    """
    return {
        'Total Queries': len(df),
        'Accuracy': df['is_correct'].mean() if 'is_correct' in df else None,
        'Avg Precision': df['precision'].mean(),
        'Avg Recall': df['recall'].mean(),
        'Avg F1 Score': df['f1_score'].mean(),
        'Confident Wrong Rate': df['confident_wrong'].mean() if 'confident_wrong' in df else None,
        'Avg Response Time (s)': df['response_time'].mean(),
        'Avg Token Usage': df['total_tokens'].mean(),
        'Total Tokens': df['total_tokens'].sum(),
        'Avg Debate Rounds': df['num_debate_rounds'].mean() if 'num_debate_rounds' in df else None,
        'Convergence Rate': df['converged'].mean() if 'converged' in df else None,
    }

# Calculate for each experiment
metrics_summary = {}
for name, df in dfs.items():
    metrics_summary[name] = calculate_metrics(df)

# Create comparison table
df_metrics = pd.DataFrame(metrics_summary).T
df_metrics = df_metrics.round(4)

print("\n" + "="*80)
print("📊 AGGREGATE METRICS COMPARISON")
print("="*80)
print(df_metrics.to_string())
print("\n")

## 3. Visualize Accuracy Comparison

In [ ]:
# Accuracy bar chart
fig, ax = plt.subplots(figsize=(10, 6))

experiments_order = ['baseline', 'baseline_updated', 'madam_old', 'madam_new']
labels = ['Baseline\n(Old Data)', 'Baseline\n(New Data)', 'MADAM-RAG\n(Old Data)', 'MADAM-RAG\n(New Data)']
colors = ['#e74c3c', '#f39c12', '#3498db', '#2ecc71']

accuracies = [metrics_summary.get(exp, {}).get('Accuracy', 0) for exp in experiments_order]

bars = ax.bar(labels, accuracies, color=colors, alpha=0.8, edgecolor='black')

# Add value labels on bars
for bar, acc in zip(bars, accuracies):
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
            f'{acc:.2%}',
            ha='center', va='bottom', fontsize=12, fontweight='bold')

ax.set_ylabel('Accuracy', fontsize=12, fontweight='bold')
ax.set_title('RAG System Accuracy Comparison (4-Way)', fontsize=14, fontweight='bold')
ax.set_ylim(0, 1.0)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.0%}'))
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('evaluation/results/accuracy_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

## 4. Precision, Recall, F1 Score Comparison

In [ ]:
# Multi-metric comparison
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

metrics_to_plot = ['Avg Precision', 'Avg Recall', 'Avg F1 Score']
titles = ['Precision', 'Recall', 'F1 Score']

for ax, metric, title in zip(axes, metrics_to_plot, titles):
    values = [metrics_summary.get(exp, {}).get(metric, 0) for exp in experiments_order]
    bars = ax.bar(labels, values, color=colors, alpha=0.8, edgecolor='black')
    
    # Add value labels
    for bar, val in zip(bars, values):
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height,
                f'{val:.3f}',
                ha='center', va='bottom', fontsize=10, fontweight='bold')
    
    ax.set_ylabel(title, fontsize=11, fontweight='bold')
    ax.set_title(f'{title} Comparison', fontsize=12, fontweight='bold')
    ax.set_ylim(0, 1.0)
    ax.set_xticklabels(labels, fontsize=9)
    ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('evaluation/results/retrieval_metrics_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

## 5. Response Time & Token Usage Comparison

In [ ]:
# Response time and token usage
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Response Time
ax1 = axes[0]
times = [metrics_summary.get(exp, {}).get('Avg Response Time (s)', 0) for exp in experiments_order]
bars1 = ax1.bar(labels, times, color=colors, alpha=0.8, edgecolor='black')
for bar, time in zip(bars1, times):
    height = bar.get_height()
    ax1.text(bar.get_x() + bar.get_width()/2., height,
            f'{time:.2f}s',
            ha='center', va='bottom', fontsize=10, fontweight='bold')
ax1.set_ylabel('Response Time (seconds)', fontsize=11, fontweight='bold')
ax1.set_title('Average Response Time', fontsize=12, fontweight='bold')
ax1.grid(axis='y', alpha=0.3)

# Token Usage
ax2 = axes[1]
tokens = [metrics_summary.get(exp, {}).get('Avg Token Usage', 0) for exp in experiments_order]
bars2 = ax2.bar(labels, tokens, color=colors, alpha=0.8, edgecolor='black')
for bar, token in zip(bars2, tokens):
    height = bar.get_height()
    ax2.text(bar.get_x() + bar.get_width()/2., height,
            f'{int(token)}',
            ha='center', va='bottom', fontsize=10, fontweight='bold')
ax2.set_ylabel('Average Tokens', fontsize=11, fontweight='bold')
ax2.set_title('Average Token Usage', fontsize=12, fontweight='bold')
ax2.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('evaluation/results/efficiency_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

## 6. Confident Wrong Analysis

In [ ]:
# Confident wrong rate (false positives with high confidence)
fig, ax = plt.subplots(figsize=(10, 6))

cw_rates = [metrics_summary.get(exp, {}).get('Confident Wrong Rate', 0) for exp in experiments_order]
bars = ax.bar(labels, cw_rates, color=colors, alpha=0.8, edgecolor='black')

for bar, rate in zip(bars, cw_rates):
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
            f'{rate:.2%}',
            ha='center', va='bottom', fontsize=12, fontweight='bold')

ax.set_ylabel('Confident Wrong Rate', fontsize=12, fontweight='bold')
ax.set_title('False Positives with High Confidence (>0.8)', fontsize=14, fontweight='bold')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.0%}'))
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('evaluation/results/confident_wrong_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

## 7. Export Results to CSV

In [ ]:
# Export aggregate metrics
df_metrics.to_csv('evaluation/results/aggregate_metrics.csv')
print("✅ Saved aggregate metrics to evaluation/results/aggregate_metrics.csv")

# Export detailed results
if 'df_all' in locals():
    df_all.to_csv('evaluation/results/detailed_results.csv', index=False)
    print("✅ Saved detailed results to evaluation/results/detailed_results.csv")

# Summary statistics
print("\n" + "="*80)
print("📈 KEY FINDINGS")
print("="*80)

if dfs:
    baseline_acc = metrics_summary.get('baseline', {}).get('Accuracy', 0)
    madam_new_acc = metrics_summary.get('madam_new', {}).get('Accuracy', 0)
    
    improvement = (madam_new_acc - baseline_acc) / baseline_acc * 100 if baseline_acc > 0 else 0
    
    print(f"Baseline Accuracy: {baseline_acc:.2%}")
    print(f"MADAM-RAG + New Data Accuracy: {madam_new_acc:.2%}")
    print(f"Improvement: +{improvement:.1f}%")
    print("\n")

## 8. Statistical Significance Testing

In [ ]:
from scipy import stats

# Compare baseline vs MADAM-RAG with t-test
if 'baseline' in dfs and 'madam_new' in dfs:
    baseline_scores = dfs['baseline']['is_correct'].dropna()
    madam_scores = dfs['madam_new']['is_correct'].dropna()
    
    # Perform t-test
    t_stat, p_value = stats.ttest_ind(baseline_scores, madam_scores)
    
    print("\n" + "="*80)
    print("📊 STATISTICAL SIGNIFICANCE TEST (Baseline vs MADAM-RAG+NewData)")
    print("="*80)
    print(f"t-statistic: {t_stat:.4f}")
    print(f"p-value: {p_value:.4f}")
    
    if p_value < 0.05:
        print("\n✅ Result: STATISTICALLY SIGNIFICANT (p < 0.05)")
        print("   The improvement from MADAM-RAG is not due to chance.")
    else:
        print("\n⚠️  Result: NOT statistically significant (p >= 0.05)")
        print("   Need more test queries or larger improvement.")
else:
    print("⚠️  Need both baseline and MADAM-RAG results for significance testing")

## 9. Generate Research Paper Table

In [ ]:
# Format for research paper
print("\n" + "="*100)
print("📄 LATEX TABLE FOR RESEARCH PAPER")
print("="*100)
print("\\begin{table}[h]")
print("\\centering")
print("\\caption{Performance Comparison of RAG Systems}")
print("\\begin{tabular}{|l|c|c|c|c|c|c|}")
print("\\hline")
print("\\textbf{System} & \\textbf{Accuracy} & \\textbf{Precision} & \\textbf{Recall} & \\textbf{F1} & \\textbf{Time (s)} & \\textbf{Tokens} \\\\")
print("\\hline")

for exp, label in zip(experiments_order, labels):
    if exp in metrics_summary:
        m = metrics_summary[exp]
        label_clean = label.replace('\n', ' ')
        print(f"{label_clean:30s} & {m['Accuracy']:.3f} & {m['Avg Precision']:.3f} & {m['Avg Recall']:.3f} & {m['Avg F1 Score']:.3f} & {m['Avg Response Time (s)']:.2f} & {int(m['Avg Token Usage'])} \\\\")

print("\\hline")
print("\\end{tabular}")
print("\\end{table}")
print("\n")